In [ ]:
#Passo 1 - importação das libraries necessárias ao projeto
import pandas as pd 
import numpy as np  
import os

caminho_sinistros = '../data/sinistros_2015-2021.csv'

**Agora, uma decisão técnica é ler apenas as primeiras 5 linhas dos documentos csv.**
Fazemos isso porque, uma vez que os documentos são pesados, não queremos carregar a memória. Por isso testamos o código com as primeiras 5 linhas apenas.

In [ ]:
#Passo 2 - Investigação do formato do documento CSV
try:
    df_amostra = pd.read_csv(caminho_sinistros, sep=';', encoding='latin1',nrows=5)
    print('Leitura com separador ; e latin1 bem-sucedida.\n')

except:
    print('Tentando com separador , e utf-8...')
    df_amostra = pd.read_csv(caminho_sinistros, nrows=5)

#Visualizar as colunas disponíveis e as primeiras linhas
print("Colunas encontradas:")
print(df_amostra.columns.tolist())

#Mostrar as primeiras linhas
df_amostra.head()

Leitura com separador ; e latin1 bem-sucedida.

Colunas encontradas:
['id_sinistro', 'tipo_registro', 'data_sinistro', 'ano_sinistro', 'mes_sinistro', 'dia_sinistro', 'hora_sinistro', 'ano_mes_sinistro', 'dia_da_semana', 'turno', 'logradouro', 'numero_logradouro', 'tipo_via', 'tipo_local', 'latitude', 'longitude', 'cod_ibge', 'municipio', 'regiao_administrativa', 'administracao', 'conservacao', 'circunscricao', 'tp_sinistro_primario', 'qtd_pedestre', 'qtd_bicicleta', 'qtd_motocicleta', 'qtd_automovel', 'qtd_onibus', 'qtd_caminhao', 'qtd_veic_outros', 'qtd_veic_nao_disponivel', 'qtd_gravidade_fatal', 'qtd_gravidade_grave', 'qtd_gravidade_leve', 'qtd_gravidade_ileso', 'qtd_gravidade_nao_disponivel', 'tp_sinistro_atrop_pedestre', 'tp_sinistro_atrop_vitima_fora_veic', 'tp_sinistro_colisao_frontal', 'tp_sinistro_colisao_traseira', 'tp_sinistro_colisao_lateral', 'tp_sinistro_colisao_transversal', 'tp_sinistro_colisao_outros', 'tp_sinistro_choque', 'tp_sinistro_atrop_animal', 'tp_sinistro_cap

,id_sinistro,tipo_registro,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,hora_sinistro,ano_mes_sinistro,dia_da_semana,turno,...,tp_sinistro_colisao_lateral,tp_sinistro_colisao_transversal,tp_sinistro_colisao_outros,tp_sinistro_choque,tp_sinistro_atrop_animal,tp_sinistro_capotamento,tp_sinistro_engavetamento,tp_sinistro_tombamento,tp_sinistro_outros,tp_sinistro_nao_disponivel
0,2489730,SINISTRO FATAL,28/12/2014,2014,12,28,NaN,2014/12,Domingo,NAO DISPONIVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S
1,2460604,SINISTRO FATAL,01/01/2015,2015,1,1,05:50,2015/01,Quinta-feira,MADRUGADA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2493691,SINISTRO FATAL,01/01/2015,2015,1,1,23:38,2015/01,Quinta-feira,NOITE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S,NaN
3,2506657,SINISTRO FATAL,01/01/2015,2015,1,1,NaN,2015/01,Quinta-feira,NAO DISPONIVEL,...,NaN,NaN,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2473612,SINISTRO FATAL,01/01/2015,2015,1,1,16:00,2015/01,Quinta-feira,TARDE,...,NaN,NaN,NaN,S,NaN,NaN,NaN,NaN,NaN,NaN


Agora, preciso descobrir como a cidade está identificad no arquivo CSV (MATAO, MATÃO, Matão ou cod_ibge = 2342334)

In [ ]:
#Passo 3.1: investigar como Matão está registrada
# Leio apenas as colunas geográficas para economizar memória e tempo

colunas_geo = ['municipio', 'cod_ibge']

df_geo = pd.read_csv(caminho_sinistros,sep=';',encoding='latin1',usecols=colunas_geo)

# 1. Procurar por variações do nome 'MATAO' ou 'MATÃO' (ignorando maiúsculas/minúsculas)
municipios_unicos = df_geo['municipio'].dropna().unique()
variacoes_matao = [m for m in municipios_unicos if 'MATA' in m.upper()]
print("Variações de nome encontradas:")
print(variacoes_matao)
# 2. Verificar se o código IBGE de Matão (3529302) está presente
ibge_matao_presente = 3529302 in df_geo['cod_ibge'].values
print(f"\nCódigo IBGE 3529302 presente? {ibge_matao_presente}")
# 3. Contagem rápida de sinistros encontrados para Matão nesse primeiro arquivo
qtd_registros = df_geo[df_geo['cod_ibge'] == 3529302].shape[0]
print(f"Total de sinistros em Matão (2015-2021): {qtd_registros}")

Variações de nome encontradas:
['MATAO']

Código IBGE 3529302 presente? True
Total de sinistros em Matão (2015-2021): 1408


In [ ]:
# Passo 3.2: Carregar todas as colunas de Matão usando leitura em lotes (chunks)
lista_lotes = []

# Lê o arquivo em pedaços de 20.000 linhas
for lote in pd.read_csv(caminho_sinistros, sep=';', encoding='latin1', chunksize=20000):
    # Filtra apenas o município de Matão pelo código IBGE
    lote_matao = lote[lote['cod_ibge'] == 3529302]
    lista_lotes.append(lote_matao)

# Concatena todos os pedaços filtrados em um único DataFrame
df_matao = pd.concat(lista_lotes, ignore_index=True)

print(f"Dimensões do DataFrame de Matão: {df_matao.shape[0]} linhas e {df_matao.shape[1]} colunas.")
print(f"Memória aproximada utilizada: {df_matao.memory_usage().sum() / 1024:.2f} KB\n")

# Exibir resumo das colunas e tipos de dados
df_matao.info()


/tmp/ipykernel_14539/3012965161.py:5: DtypeWarning: Columns (37: tp_sinistro_atrop_vitima_fora_veic) have mixed types. Specify dtype option on import or set low_memory=False.
  for lote in pd.read_csv(caminho_sinistros, sep=';', encoding='latin1', chunksize=20000):


Dimensões do DataFrame de Matão: 1408 linhas e 50 colunas.
Memória aproximada utilizada: 550.13 KB

<class 'pandas.DataFrame'>
RangeIndex: 1408 entries, 0 to 1407
Data columns (total 50 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id_sinistro                         1408 non-null   int64  
 1   tipo_registro                       1408 non-null   str    
 2   data_sinistro                       1408 non-null   str    
 3   ano_sinistro                        1408 non-null   int64  
 4   mes_sinistro                        1408 non-null   int64  
 5   dia_sinistro                        1408 non-null   int64  
 6   hora_sinistro                       1407 non-null   str    
 7   ano_mes_sinistro                    1408 non-null   str    
 8   dia_da_semana                       1408 non-null   str    
 9   turno                               1408 non-null   str    
 10  logradouro         

In [ ]:
# Passo 3.3: Inspecionar o conteúdo das principais variáveis

print("--- 1. Tipo de Registro (Geral) ---")
print(df_matao['tipo_registro'].value_counts(dropna=False))

print("\n--- 2. Top 5 Tipos Primários de Sinistro ---")
print(df_matao['tp_sinistro_primario'].value_counts().head(5))

print("\n--- 3. Amostra de Coordenadas (Latitude / Longitude) ---")
print(df_matao[['latitude', 'longitude']].dropna().head(5))

print("\n--- 4. Resumo das Vítimas e Gravidade (Valores não nulos) ---")
colunas_gravidade = [
    'qtd_gravidade_fatal', 
    'qtd_gravidade_grave', 
    'qtd_gravidade_leve', 
    'qtd_gravidade_ileso'
]
print(df_matao[colunas_gravidade].describe())


--- 1. Tipo de Registro (Geral) ---
tipo_registro
SINISTRO NAO FATAL    697
NOTIFICACAO           639
SINISTRO FATAL         72
Name: count, dtype: int64

--- 2. Top 5 Tipos Primários de Sinistro ---
tp_sinistro_primario
OUTROS            629
COLISAO           436
NAO DISPONIVEL    159
CHOQUE            124
ATROPELAMENTO      60
Name: count, dtype: int64

--- 3. Amostra de Coordenadas (Latitude / Longitude) ---
        latitude      longitude
0  -21,598427248  -48,362947675
1      -21,66172      -48,37225
2  -21,627512859  -48,383316368
3     -21,593718     -48,384002
4  -21,629461832   -48,33564302

--- 4. Resumo das Vítimas e Gravidade (Valores não nulos) ---
       qtd_gravidade_fatal  qtd_gravidade_grave  qtd_gravidade_leve  \
count            72.000000            94.000000          624.000000   
mean              1.069444             1.063830            1.221154   
std               0.255992             0.286188            0.533742   
min               1.000000             1.00000

In [ ]:
# Passo 3.4 (conclusão): Distribuição temporal e auditoria de dados faltantes

print("--- Distribuição de Sinistros por Ano (2015-2021) ---")
print(df_matao['ano_sinistro'].value_counts().sort_index())

print("\n--- Percentual de Dados Faltantes por Coluna (Top 15 mais vazias) ---")
nulos_pct = (df_matao.isnull().mean() * 100).sort_values(ascending=False)
print(nulos_pct.head(15).round(2).astype(str) + '%')

print("\n--- Total de colunas 100% preenchidas ---")
colunas_completas = df_matao.columns[df_matao.notnull().all()].tolist()
print(f"Total: {len(colunas_completas)} colunas sem nenhum nulo.")
print(colunas_completas[:10], "... (veja as 10 primeiras)")


--- Distribuição de Sinistros por Ano (2015-2021) ---
ano_sinistro
2015     12
2016     14
2017      6
2018     10
2019    470
2020    410
2021    486
Name: count, dtype: int64

--- Percentual de Dados Faltantes por Coluna (Top 15 mais vazias) ---
tp_sinistro_atrop_vitima_fora_veic    100.0%
tp_sinistro_colisao_traseira          100.0%
tp_sinistro_engavetamento             99.93%
tp_sinistro_atrop_animal              99.29%
qtd_onibus                            99.22%
tp_sinistro_colisao_outros            99.01%
qtd_gravidade_nao_disponivel          98.08%
qtd_bicicleta                         97.73%
qtd_pedestre                           97.3%
qtd_veic_outros                        96.8%
tp_sinistro_capotamento               96.59%
tp_sinistro_tombamento                95.88%
tp_sinistro_atrop_pedestre            95.74%
qtd_gravidade_fatal                   94.89%
qtd_gravidade_grave                   93.32%
dtype: str

--- Total de colunas 100% preenchidas ---
Total: 15 colunas sem n

Por que isso aconteceu? Matão teve uma "explosão" de acidentes em 2019?

**Não! Trata-se de uma mudança metodológica do Infosiga SP.**

Até meados de 2018/2019, o Infosiga registrava apenas sinistros fatais (aqueles com óbito confirmado).
A partir de 2019, o sistema passou a integrar os dados do Corpo de Bombeiros, SAMU e Polícia Militar, passando a registrar também os sinistros não fatais e notificações.
Se alguém fizesse um gráfico direto sem saber disso, diria erroneamente que os acidentes explodiram 4.000% em 2019. Na verdade, apenas a forma de coletar os dados mudou!